# 🥭 Mangosteen Dataset Preparation Notebook

**COE67-312 · AI in Embedded Systems · Mini-Project Data Prep**

This notebook prepares the dataset for training. **Data Sub-Team runs this.**

## What this notebook does

1. **Load** the 245 raw images from Google Drive
2. **Label** each image (Unripe / Ripe / Overripe / Skip) - you click a button
3. **Auto-crop** each image to isolate the mangosteen (removes ruler, red marker, background)
4. **Verify** the crops in a gallery - reject bad ones
5. **Split** into train / val / test sets
6. **Save** the final dataset ready for the Model Sub-Team

## How long it takes

About **45 minutes** for the whole Data Sub-Team working together.

- Labelling: ~15 min (~4 sec per image)
- Auto-crop: ~5 min (runs automatically)
- Verification: ~15 min (visual check)
- Split + save: ~10 min

## Before you start

1. **Get the shared Google Drive link** from your instructor
2. **Add the shared folder to your Drive** (right-click → Add shortcut to Drive)
3. **Run this notebook top to bottom** — do not skip cells


## Step 1 — Setup

Run this cell first. It installs packages and mounts your Google Drive.

In [ ]:
# Install required packages
!pip install -q ipywidgets opencv-python-headless numpy matplotlib

import os, shutil, json, random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
import ipywidgets as widgets

from google.colab import drive
drive.mount('/content/drive')

print('Setup complete.')

## Step 2 — Set paths

**Edit `RAW_IMAGES_DIR` below** to point to your shared folder in Google Drive.

Example: `'/content/drive/MyDrive/COE67-312/mangosteen_raw'`

In [ ]:
# ============ EDIT THIS ============
RAW_IMAGES_DIR = '/content/drive/MyDrive/COE67-312/mangosteen_raw'
# ===================================

# Where cleaned dataset will be saved (in your Drive so Model Sub-Team can access)
OUTPUT_DIR = '/content/drive/MyDrive/COE67-312/mangosteen_dataset'

# Local working directory (fast)
WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

# Verify the raw folder exists
raw_path = Path(RAW_IMAGES_DIR)
if not raw_path.exists():
    raise FileNotFoundError(f'Cannot find {RAW_IMAGES_DIR}. Fix the path above.')

# List all images
extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
raw_files = sorted([f for f in raw_path.iterdir() if f.suffix in extensions])
print(f'Found {len(raw_files)} raw images')
if len(raw_files) < 100:
    print('⚠️  Expected around 245 images. Check the folder path.')

## Step 3 — Auto-crop function (Hough Circle Detection)

This function finds the round mangosteen in each image using Hough Circle Transform.
It works well because mangosteens are round. Ruler and red marker are ignored.

**You do not need to modify this cell** — just run it.

In [ ]:
def auto_crop_mangosteen(img_path, target_size=96):
    """
    Detect the round mangosteen in an image and crop a square around it.
    Returns (cropped_96x96, cropped_full_size, success) tuple.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None, None, False
    h, w = img.shape[:2]

    # Convert to HSV to mask out non-fruit regions
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hue, sat, val = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]

    # Mask: blue ruler and red marker square (so Hough ignores them)
    is_blue = (hue >= 95) & (hue <= 135) & (sat > 40)
    is_red = ((hue <= 8) | (hue >= 172)) & (sat > 180) & (val > 150)

    # Grayscale with non-fruit masked white
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_masked = gray.copy()
    gray_masked[is_blue | is_red] = 255
    gray_blurred = cv2.medianBlur(gray_masked, 15)

    # Hough circles — fruit is round, this is a natural fit
    min_dim = min(h, w)
    circles = cv2.HoughCircles(
        gray_blurred, cv2.HOUGH_GRADIENT,
        dp=1.5, minDist=min_dim // 2,
        param1=100, param2=30,
        minRadius=int(min_dim * 0.05),
        maxRadius=int(min_dim * 0.35))

    if circles is None:
        return None, None, False

    circles = np.round(circles[0]).astype(int)
    cx, cy, r = circles[0]

    # Crop square with 20% padding
    pad = int(r * 0.20)
    side = 2 * (r + pad)
    x1 = max(0, cx - side // 2)
    y1 = max(0, cy - side // 2)
    x2 = min(w, x1 + side)
    y2 = min(h, y1 + side)
    if x2 - x1 < side: x1 = max(0, x2 - side)
    if y2 - y1 < side: y1 = max(0, y2 - side)

    crop_full = img[y1:y2, x1:x2]
    if crop_full.size == 0:
        return None, None, False

    crop_small = cv2.resize(crop_full, (target_size, target_size),
                             interpolation=cv2.INTER_AREA)
    return crop_small, crop_full, True

print('Auto-crop function ready.')

## Step 4 — Test auto-crop on the first 6 images

Run this to verify auto-crop is working on your images. You should see 6 nicely cropped mangosteens.

In [ ]:
# Test on first 6 images
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, f in enumerate(raw_files[:6]):
    img = cv2.imread(str(f))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img_rgb)
    axes[0, i].set_title(f'Original: {f.name[:20]}', fontsize=8)
    axes[0, i].axis('off')

    crop_small, crop_full, ok = auto_crop_mangosteen(f)
    if ok:
        crop_rgb = cv2.cvtColor(crop_small, cv2.COLOR_BGR2RGB)
        axes[1, i].imshow(crop_rgb)
        axes[1, i].set_title('Auto-crop OK', fontsize=8, color='green')
    else:
        axes[1, i].text(0.5, 0.5, 'FAILED', ha='center', va='center',
                        color='red', fontsize=14)
        axes[1, i].set_title('Crop failed', fontsize=8, color='red')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

print('If most crops look good, continue. If most fail, ask instructor for help.')

## Step 5 — Label all images

This shows each image one by one. Click the correct button:

- **🟢 Ripe** — deep purple, uniformly dark, sepals still green
- **🟡 Unripe** — greenish-red or yellow-green, firm shell
- **🔴 Overripe** — very dark, dull patches, dried sepals, OR mushy
- **⏭️ Skip** — blurry, hand in frame, no fruit visible, damaged photo

**Tip:** split the 245 images among 5 team members = ~50 per person = ~4 minutes each.

Your progress is saved to `labels.json` after every click. If Colab disconnects, just re-run the cell.

In [ ]:
# Load existing labels if any (for resuming)
labels_file = f'{WORK_DIR}/labels.json'
labels = {}
if os.path.exists(labels_file):
    with open(labels_file) as f:
        labels = json.load(f)
    print(f'Resuming - {len(labels)} images already labelled')

# Find next unlabelled image
todo = [f for f in raw_files if f.name not in labels]
print(f'Images to label: {len(todo)} of {len(raw_files)}')

# State
state = {'idx': 0, 'todo': todo}

# Widgets
image_output = widgets.Output()
progress = widgets.HTML(value='')

def save_labels():
    with open(labels_file, 'w') as f:
        json.dump(labels, f, indent=2)

def show_current():
    with image_output:
        clear_output(wait=True)
        if state['idx'] >= len(state['todo']):
            print('🎉 All done! Move to Step 6.')
            return
        img_path = state['todo'][state['idx']]
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Downsize for display
        h, w = img_rgb.shape[:2]
        scale = min(500 / w, 500 / h)
        img_small = cv2.resize(img_rgb, (int(w*scale), int(h*scale)))
        plt.figure(figsize=(6, 6))
        plt.imshow(img_small)
        plt.axis('off')
        plt.title(img_path.name, fontsize=10)
        plt.show()
    total = len(raw_files)
    done = len(labels)
    progress.value = f'<b>Labelled: {done} / {total}</b> ({done*100//total}%)'

def make_click(label):
    def click(_):
        if state['idx'] < len(state['todo']):
            img_name = state['todo'][state['idx']].name
            labels[img_name] = label
            save_labels()
            state['idx'] += 1
            show_current()
    return click

btn_unripe = widgets.Button(description='🟡 Unripe', button_style='warning',
                             layout=widgets.Layout(width='120px', height='40px'))
btn_ripe   = widgets.Button(description='🟢 Ripe',   button_style='success',
                             layout=widgets.Layout(width='120px', height='40px'))
btn_over   = widgets.Button(description='🔴 Overripe', button_style='danger',
                             layout=widgets.Layout(width='120px', height='40px'))
btn_skip   = widgets.Button(description='⏭️ Skip',   button_style='',
                             layout=widgets.Layout(width='120px', height='40px'))

btn_unripe.on_click(make_click('unripe'))
btn_ripe.on_click(make_click('ripe'))
btn_over.on_click(make_click('overripe'))
btn_skip.on_click(make_click('skip'))

buttons = widgets.HBox([btn_unripe, btn_ripe, btn_over, btn_skip])
display(progress, image_output, buttons)
show_current()

## Step 6 — Auto-crop all labelled images

Once labelling is complete, run this to crop every image using Hough circle detection.

This takes about 3-5 minutes for 245 images.

In [ ]:
# Load labels
with open(f'{WORK_DIR}/labels.json') as f:
    labels = json.load(f)

print(f'Loaded {len(labels)} labels')
print('Distribution:')
for cls in ['unripe', 'ripe', 'overripe', 'skip']:
    count = sum(1 for v in labels.values() if v == cls)
    print(f'  {cls:10}: {count}')

# Prepare output structure
for cls in ['unripe', 'ripe', 'overripe']:
    os.makedirs(f'{WORK_DIR}/cropped/{cls}', exist_ok=True)
os.makedirs(f'{WORK_DIR}/crop_failed', exist_ok=True)

# Process each image
crop_results = {'ok': 0, 'failed': 0, 'skipped': 0}
failed_files = []

for i, f in enumerate(raw_files):
    label = labels.get(f.name, 'skip')
    if label == 'skip':
        crop_results['skipped'] += 1
        continue
    crop_small, crop_full, ok = auto_crop_mangosteen(f)
    if ok:
        out_path = f'{WORK_DIR}/cropped/{label}/{f.stem}.jpg'
        cv2.imwrite(out_path, crop_small)
        crop_results['ok'] += 1
    else:
        # Save original to failed folder for manual review
        shutil.copy(str(f), f'{WORK_DIR}/crop_failed/{f.name}')
        failed_files.append(f.name)
        crop_results['failed'] += 1
    if (i + 1) % 25 == 0:
        print(f'  Processed {i+1} / {len(raw_files)}')

print()
print(f"✅ Cropped OK: {crop_results['ok']}")
print(f"❌ Crop failed: {crop_results['failed']}")
print(f"⏭️  Skipped: {crop_results['skipped']}")

if crop_results['failed'] > 0:
    print(f'\nFailed files (crop manually or discard):')
    for name in failed_files[:10]:
        print(f'  - {name}')
    if len(failed_files) > 10:
        print(f'  ... and {len(failed_files) - 10} more')

## Step 7 — Verify crops in a gallery

Look at 20 random crops per class. Any that don't look like mangosteens should be manually deleted from the folders in `/content/work/cropped/{class}`.

Common problems to look for:
- Cropped ruler instead of fruit
- Cropped only the sepal (top leaves)
- Half fruit, half background

In [ ]:
import random

for cls in ['unripe', 'ripe', 'overripe']:
    folder = Path(f'{WORK_DIR}/cropped/{cls}')
    files = sorted(folder.glob('*.jpg'))
    if not files:
        print(f'No files in {cls}')
        continue
    sample = random.sample(files, min(20, len(files)))

    print(f'\n=== {cls.upper()} ({len(files)} total) ===')
    cols = 5
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
    axes = axes.flatten() if rows > 1 else axes
    for i, f in enumerate(sample):
        img = cv2.imread(str(f))
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f.name[:15], fontsize=7)
        axes[i].axis('off')
    for i in range(len(sample), len(axes)):
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

## Step 8 — Delete bad crops (optional)

If you saw bad crops in the gallery above, delete them manually.

**Method A - File browser:** Click the folder icon on the left sidebar, navigate to `/content/work/cropped/<class>/`, right-click bad files → Delete.

**Method B - Delete by filename in code cell below.**

In [ ]:
# Delete specific bad crops by name. Add file names to the lists below.

bad_unripe = [
    # 'IMG20230903161532.jpg',
]
bad_ripe = [
    # e.g. '20230903_170920.jpg',
]
bad_overripe = [
    # e.g. '20230903_163050.jpg',
]

for cls, badlist in [('unripe', bad_unripe), ('ripe', bad_ripe), ('overripe', bad_overripe)]:
    for name in badlist:
        path = f'{WORK_DIR}/cropped/{cls}/{name}'
        if os.path.exists(path):
            os.remove(path)
            print(f'Deleted {path}')

# Show updated counts
for cls in ['unripe', 'ripe', 'overripe']:
    count = len(list(Path(f'{WORK_DIR}/cropped/{cls}').glob('*.jpg')))
    print(f'{cls}: {count} images')

## Step 9 — Train / Val / Test split

Split each class 70 / 15 / 15% into train, validation, and test sets.

**Important:** we use a fixed random seed so both Group 1 and Group 2 have IDENTICAL splits.
This makes their final accuracy numbers comparable.

In [ ]:
SEED = 42
random.seed(SEED)

# Prepare output structure
final_dir = Path(f'{WORK_DIR}/final_dataset')
if final_dir.exists():
    shutil.rmtree(final_dir)
for split in ['train', 'val', 'test']:
    for cls in ['unripe', 'ripe', 'overripe']:
        (final_dir / split / cls).mkdir(parents=True, exist_ok=True)

# Split each class independently to keep balance
split_summary = []
for cls in ['unripe', 'ripe', 'overripe']:
    files = sorted(Path(f'{WORK_DIR}/cropped/{cls}').glob('*.jpg'))
    random.shuffle(files)
    n = len(files)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)
    # test = rest

    train_files = files[:n_train]
    val_files   = files[n_train:n_train + n_val]
    test_files  = files[n_train + n_val:]

    for f in train_files: shutil.copy(str(f), str(final_dir / 'train' / cls / f.name))
    for f in val_files:   shutil.copy(str(f), str(final_dir / 'val'   / cls / f.name))
    for f in test_files:  shutil.copy(str(f), str(final_dir / 'test'  / cls / f.name))

    split_summary.append([cls, len(train_files), len(val_files), len(test_files), n])

# Print summary
print(f'{"Class":10}  {"Train":>6}  {"Val":>6}  {"Test":>6}  {"Total":>7}')
print('-' * 46)
totals = [0, 0, 0, 0]
for row in split_summary:
    print(f'{row[0]:10}  {row[1]:>6}  {row[2]:>6}  {row[3]:>6}  {row[4]:>7}')
    for i in range(4): totals[i] += row[i+1] if i < 3 else row[4]
print('-' * 46)
print(f'{"TOTAL":10}  {totals[0]:>6}  {totals[1]:>6}  {totals[2]:>6}  {totals[3]:>7}')

## Step 10 — Save final dataset to Google Drive

This copies the split dataset to your shared Drive folder so the Model Sub-Team can access it.

In [ ]:
print(f'Copying to {OUTPUT_DIR}...')
if os.path.exists(OUTPUT_DIR):
    print('Removing old dataset...')
    shutil.rmtree(OUTPUT_DIR)
shutil.copytree(f'{WORK_DIR}/final_dataset', OUTPUT_DIR)

# Also save the labels.json for reference
shutil.copy(f'{WORK_DIR}/labels.json', f'{OUTPUT_DIR}/labels.json')

print(f'\n✅ Dataset ready at: {OUTPUT_DIR}')
print(f'\nFolder structure:')
for split in ['train', 'val', 'test']:
    for cls in ['unripe', 'ripe', 'overripe']:
        path = Path(OUTPUT_DIR) / split / cls
        count = len(list(path.glob('*.jpg')))
        print(f'  {OUTPUT_DIR}/{split}/{cls}/  ({count} images)')

print(f'\nHand this Drive path to the Model Sub-Team.')

## 🎉 Done!

**What to tell the Model Sub-Team:**

> The dataset is in Google Drive at `/content/drive/MyDrive/COE67-312/mangosteen_dataset`
>
> Structure: `train/{unripe,ripe,overripe}/` + `val/...` + `test/...`
>
> Images are 96×96 RGB JPG, already cropped to the fruit.
>
> Both groups have identical splits (seed=42) so accuracy numbers are comparable.

---

**Data Sub-Team responsibility ends here.** Now the Model Sub-Team takes over with the training notebook.